## Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.
Messages are objects that contain:
 - Role - Identifies the message type (e.g. system, user)
 - Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 - Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL")
model = ChatOpenAI(
    model="gpt-4.1",
    api_key = openai_api_key,
    base_url = openai_base_url
)
model

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000020E734ECAD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000020E734ED6A0>, root_client=<openai.OpenAI object at 0x0000020E7301D550>, root_async_client=<openai.AsyncOpenAI object at 0x0000020E734ED400>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://yibuapi.com/v1')

## Text Prompts
Text prompts are strings - ideal for straightforward generation tasks where you don’t need to retain conversation history.

## Message Prompts
Alternatively, you can pass in a list of messages to the model by providing a list of message objects.


Message types
- System message - Tells the model how to behave and provide context for interactions
- Human message - Represents user input and interactions with the model
- AI message - Responses generated by the model, including text content, tool calls, and metadata
- Tool message - Represents the outputs of tool calls

## System Message
A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.


## Human Message
A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

## AI Message
An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

## Tool Message
For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [3]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What a poem!")
]

response = model.invoke(messages)
response.content

"Thank you! If you'd like, I can write a poem for you or discuss the poem you're referring to. Let me know how I can help!"

In [5]:

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

To create a REST API in Python, the most common web frameworks are Flask and FastAPI. Here’s a concise guide using both:

---

### Using **Flask**

1. **Install Flask:**
   ```bash
   pip install Flask
   ```

2. **Basic REST API Example:**
   ```python
   from flask import Flask, request, jsonify

   app = Flask(__name__)

   # Example data store
   items = []

   # Get all items
   @app.route('/items', methods=['GET'])
   def get_items():
       return jsonify(items)

   # Add an item
   @app.route('/items', methods=['POST'])
   def add_item():
       data = request.json
       items.append(data)
       return jsonify(data), 201

   if __name__ == '__main__':
       app.run(debug=True)
   ```
   **Explanation:**  
   - `GET /items` returns the list of items.  
   - `POST /items` accepts JSON and appends it to the items list.

---

### Using **FastAPI** (recommended for modern APIs)

1. **Install FastAPI and Uvicorn:**
   ```bash
   pip install fastapi uvicorn
   ```

2. **Basic REST 

In [8]:
human_msg = HumanMessage(
    content = "Hello",
    name = "awu",
    id = "msg_222"
)

response = model.invoke([human_msg])
print(response.content)

Hello! How can I help you today?


In [9]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

ai_msg = AIMessage("I'd be happy to help you with that question!")

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("Can you help me with something?"),
    ai_msg,
    HumanMessage("Great! What's 2 + 2?")
]

response = model.invoke(messages)
response.content

'2 + 2 equals 4. If you have any other questions, feel free to ask!'

In [10]:
response.usage_metadata

{'input_tokens': 51,
 'output_tokens': 21,
 'total_tokens': 72,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [12]:
from langchain.messages import AIMessage, ToolMessage

ai_messages = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "New York"},
        "id": "call_222"
    }]
)

weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_222"
)

messages = [
    HumanMessage("What is the weather like in New York?"),
    ai_messages,
    tool_message
]

response = model.invoke(messages)
response

AIMessage(content='The weather in New York is currently sunny with a temperature of 72°F.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 44, 'total_tokens': 61, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'input_tokens': 0, 'input_tokens_details': None, 'output_tokens': 0}, 'model_provider': 'openai', 'model_name': 'gpt-4.1', 'system_fingerprint': 'fp_7a7fd0eb44', 'id': 'chatcmpl-DK1enPuBRJ6Yia412wrHz98ZaDbVK', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf6b0-ae2c-7791-bcb1-d8393d74d205-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 44, 'output_tokens': 17, 'total_tokens': 61, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_222')